In [1]:
import sympy as sp
from scipy.optimize import fsolve
import sympy.physics.mechanics as me 
sp.init_printing(use_latex="mathjax")
me.init_vprinting()
from IPython.display import display

## Frames, Points, coordinates

In [2]:
N, B, U1, U2, L1, L2 = sp.symbols('N, B, U_1, U_2, L_1, L_2', cls=me.ReferenceFrame)
O, T, C1, C2, R1, R2 = sp.symbols('O, T, C_1, C_2, R_1, R_2', cls=me.Point)                                 # CoM points for the bodies
A1, A2, B1, B2, K1, K2, O1, O2 = sp.symbols('A_1, A_2, B_1, B_2, K_1, K_2, O_1, O_2', cls=me.Point)         # points for constraints / loop closure
alpha, beta = me.dynamicsymbols('alpha beta', real="True")
theta1, theta2 = me.dynamicsymbols('theta1:3', real="True")
psi1, psi2, psi3, psi4, psi5, psi6 = me.dynamicsymbols('psi1:7', real="True")
lt, ls, lc, lr, lf = sp.symbols('l_t, l_s, l_c, l_r, l_f' , positive="True", real="True")
m, g = sp.symbols('m, g', real="True", positive="True")
t = me.dynamicsymbols._t


p = sp.Matrix([ lt, ls, lc, lr, lf, m, g])
u1, u2, u3, u4, u5, u6, u7, u8, u9, u10 = me.dynamicsymbols('u1:11')
q = sp.Matrix([theta1, theta2])
q_r = sp.Matrix([alpha, beta, psi1, psi2, psi3, psi4, psi5, psi6])
q_N = q.col_join(q_r)

u = sp.Matrix([u1, u2])
u_r = sp.Matrix([u3, u4, u5, u6, u7, u8, u9, u10])
u_N = u.col_join(u_r)

qdot_N = q_N.diff(t)
udot = u.diff(t)

q_N_zero = {q:0 for q in q_N}
u_r_zero = {u: 0 for u in u_r}
u_N_zero = {u: 0 for u in u_N}
qdot_N_zero = {qd: 0 for qd in qdot_N}
u_d_zero = {ud: 0 for ud in udot}

q, q_r, q_N, u, u_r, u_N, qdot_N, udot

⎛            ⎡θ₁⎤               ⎡u₁ ⎤  ⎡θ₁̇⎤      ⎞
⎜            ⎢  ⎥               ⎢   ⎥  ⎢  ⎥      ⎟
⎜      ⎡α ⎤  ⎢θ₂⎥        ⎡u₃ ⎤  ⎢u₂ ⎥  ⎢θ₂̇⎥      ⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢   ⎥  ⎢   ⎥  ⎢  ⎥      ⎟
⎜      ⎢β ⎥  ⎢α ⎥        ⎢u₄ ⎥  ⎢u₃ ⎥  ⎢α̇ ⎥      ⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢   ⎥  ⎢   ⎥  ⎢  ⎥      ⎟
⎜      ⎢ψ₁⎥  ⎢β ⎥        ⎢u₅ ⎥  ⎢u₄ ⎥  ⎢β̇ ⎥      ⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢   ⎥  ⎢   ⎥  ⎢  ⎥      ⎟
⎜⎡θ₁⎤  ⎢ψ₂⎥  ⎢ψ₁⎥  ⎡u₁⎤  ⎢u₆ ⎥  ⎢u₅ ⎥  ⎢ψ₁̇⎥  ⎡u₁̇⎤⎟
⎜⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢  ⎥, ⎢   ⎥, ⎢   ⎥, ⎢  ⎥, ⎢  ⎥⎟
⎜⎣θ₂⎦  ⎢ψ₃⎥  ⎢ψ₂⎥  ⎣u₂⎦  ⎢u₇ ⎥  ⎢u₆ ⎥  ⎢ψ₂̇⎥  ⎣u₂̇⎦⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢   ⎥  ⎢   ⎥  ⎢  ⎥      ⎟
⎜      ⎢ψ₄⎥  ⎢ψ₃⎥        ⎢u₈ ⎥  ⎢u₇ ⎥  ⎢ψ₃̇⎥      ⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢   ⎥  ⎢   ⎥  ⎢  ⎥      ⎟
⎜      ⎢ψ₅⎥  ⎢ψ₄⎥        ⎢u₉ ⎥  ⎢u₈ ⎥  ⎢ψ₄̇⎥      ⎟
⎜      ⎢  ⎥  ⎢  ⎥        ⎢   ⎥  ⎢   ⎥  ⎢  ⎥      ⎟
⎜      ⎣ψ₆⎦  ⎢ψ₅⎥        ⎣u₁₀⎦  ⎢u₉ ⎥  ⎢ψ₅̇⎥      ⎟
⎜            ⎢  ⎥               ⎢   ⎥  ⎢  ⎥      ⎟
⎝            ⎣ψ₆⎦               ⎣u₁₀⎦  ⎣ψ₆̇⎦      ⎠

## Framing and coordinates setting

In [3]:
B.orient_body_fixed(N, (alpha, beta, 0), 'XYZ')
U1.orient_axis(B, B.y, theta1)
L1.orient_body_fixed(U1, (psi1, psi2, psi3), 'ZXZ')

U2.orient_axis(B, B.y, theta2)
L2.orient_body_fixed(U2, (psi4, psi5, psi6), 'ZXZ')

O1.set_pos(O, lc*N.x + ls/2*N.y - lf*N.z)
O2.set_pos(O, lc*N.x - ls/2*N.y - lf*N.z)
T.set_pos(O, lt*B.z)

A1.set_pos(O, ls/2*B.y + (lr-lf)*B.z)
C1.set_pos(O, lc/2*U1.x + ls/2*U1.y + (lr-lf)*U1.z)
B1.set_pos(C1, lc/2*U1.x)
R1.set_pos(B1,- lr/2*L1.z)
K1.set_pos(R1,- lr/2*L1.z)


A2.set_pos(O,- ls/2*B.y + (lr-lf)*B.z)
C2.set_pos(O, lc/2*U2.x - ls/2*U2.y + (lr-lf)*U2.z)
B2.set_pos(C2, lc/2*U2.x)
R2.set_pos(B2,- lr/2*L2.z)
K2.set_pos(R2,- lr/2*L2.z)


In [4]:
# position check

K1.pos_from(O).express(N).xreplace(q_N_zero)

In [5]:
# Position dependency

me.find_dynamicsymbols(T.pos_from(O), reference_frame=N), me.find_dynamicsymbols(C1.pos_from(O), reference_frame=N), me.find_dynamicsymbols(C2.pos_from(O), reference_frame=N), me.find_dynamicsymbols(R1.pos_from(O), reference_frame=N),me.find_dynamicsymbols(K1.pos_from(O), reference_frame=N), me.find_dynamicsymbols(R2.pos_from(O), reference_frame=N),  me.find_dynamicsymbols(K2.pos_from(O), reference_frame=N)

In [6]:
# loop closure equations

loop_closure1 = K1.pos_from(O) - O1.pos_from(O)
loop_closure2 = K2.pos_from(O) - O2.pos_from(O)
rod_constraint1 = B1.pos_from(O) - O1.pos_from(O)
rod_constraint2 = B2.pos_from(O) - O2.pos_from(O)

constraints = [
    loop_closure1,
    loop_closure2,
    rod_constraint1,
    rod_constraint2
    ]

# for c in constraints:
#     display(c.express(N))


## Algebraic Holonomic constraint equations

In [7]:
# Holonomic constraints

fh = sp.Matrix([
    loop_closure1.dot(N.y) - 0,
    loop_closure2.dot(N.y) - 0,
    loop_closure1.dot(N.z) - 0,
    loop_closure2.dot(N.z) - 0,
    rod_constraint1.dot(rod_constraint1) - lr**2,
    rod_constraint2.dot(rod_constraint2) - lr**2,
    # loop_closure1.dot(N.x) - 0,
    # loop_closure2.dot(N.x) - 0,
    ])

fh = fh.applyfunc(lambda i: sp.simplify(sp.trigsimp(i)))
fh


⎡                                                                              ↪
⎢                                                                   l_c⋅sin(β  ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                   l_c⋅sin(β  ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                  -l_c⋅sin(β  ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

#### Constraint equation dependents

In [8]:
me.find_dynamicsymbols(fh[0]), me.find_dynamicsymbols(fh[1]),


In [9]:
me.find_dynamicsymbols(fh[2]), me.find_dynamicsymbols(fh[3]),
    

In [10]:
me.find_dynamicsymbols(fh[4]), me.find_dynamicsymbols(fh[5])

In [11]:
fh[4].free_symbols, fh[5].free_symbols

## Kinematic differential equations

In [12]:
fk = sp.Matrix([
    theta1.diff(t) - u1,
    theta2.diff(t) - u2,
    alpha.diff(t) - u3,
    beta.diff(t) - u4,
    psi1.diff(t) - u5,
    psi2.diff(t) - u6,
    psi3.diff(t) - u7,
    psi4.diff(t) - u8,
    psi5.diff(t) - u9,
    psi6.diff(t) - u10,
])

Mk = fk.jacobian(qdot_N)
gk = fk.xreplace(qdot_N_zero)
qdot_N_sol = -Mk.LUsolve(gk)

qdot_N_replace = dict(zip(qdot_N, qdot_N_sol))
qdot_N_replace

In [13]:
fh_d = fh.diff(t).xreplace(qdot_N_replace)
fh_d

⎡                                                                              ↪
⎢                    l_c⋅(u₁ + u₄)⋅sin(α)⋅cos(β + θ₁) + l_c⋅u₃⋅sin(β + θ₁)⋅cos ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                    l_c⋅(u₂ + u₄)⋅sin(α)⋅cos(β + θ₂) + l_c⋅u₃⋅sin(β + θ₂)⋅cos ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                   -l_c⋅(u₁ + u₄)⋅cos(β + θ₁)⋅cos(α) + l_c⋅u₃⋅sin(β + θ₁)⋅sin ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

In [14]:
me.find_dynamicsymbols(fh_d)

In [18]:
Mh_d = fh_d.jacobian(u_r)
gh_d = fh_d.xreplace(u_r_zero)
u_r_sol = -Mh_d.LUsolve(gh_d)

NotImplementedError: Underdetermined systems not supported.